**Install Dependencies**

In [ ]:
!pip install biopython pandas requests tqdm

**Main Script**

Upload your Input file labelled as Spectronaut.tsv into the session storage and run the next block.

In [ ]:
# ============================================
# Spectronaut Peptide Property Analyzer
# ============================================

!pip install biopython pandas tqdm

import pandas as pd
import re
from collections import Counter
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from tqdm import tqdm

# ============================================
# CONFIG
# ============================================

INPUT_FILE = "Spectronaut.tsv"
OUTPUT_FILE = "output.tsv"

PROTON_MASS = 1.007276466812

# ============================================
# COMMON PTM MASS SHIFTS (Da)
# Monoisotopic deltas
# ============================================

MOD_MASSES = {
    "Carbamidomethyl": 57.021464,
    "Oxidation": 15.994915,
    "Phospho": 79.966331,
    "Acetyl": 42.010565,
    "Deamidated": 0.984016,
    "Pyro-glu": -17.026549,
    "Gln->pyro-Glu": -17.026549,
    "Glu->pyro-Glu": -18.010565,
    "GlyGly": 114.042927,
    "TMT6plex": 229.162932,
    "TMTpro": 304.207146,
    "Label:13C(6)15N(2)": 8.014199,
    "Label:13C(6)15N(4)": 10.008269,
}

# ============================================
# GLOBAL PTM COUNTERS
# ============================================

ptm_counter = Counter()
unknown_ptm_counter = Counter()

# ============================================
# FUNCTIONS
# ============================================

def parse_precursor(precursor):

    precursor = str(precursor)

    # -------------------------
    # Charge state
    # -------------------------
    charge_match = re.search(r"\.(\d+)$", precursor)

    charge = None
    if charge_match:
        charge = int(charge_match.group(1))

    precursor_nocharge = re.sub(r"\.\d+$", "", precursor)

    # -------------------------
    # Remove flanking underscores
    # -------------------------
    precursor_nocharge = precursor_nocharge.replace("_", "")

    # -------------------------
    # Find modifications
    # -------------------------
    raw_mods = re.findall(r"\[([^\]]+)\]", precursor_nocharge)

    modifications = []
    unknown_modifications = []

    mod_mass_delta = 0.0

    for mod in raw_mods:

        # Example:
        # Carbamidomethyl (C)

        mod_name_match = re.match(r"^([^(]+)", mod)

        if mod_name_match:
            mod_name = mod_name_match.group(1).strip()
        else:
            mod_name = mod.strip()

        modifications.append(mod_name)

        ptm_counter[mod_name] += 1

        if mod_name in MOD_MASSES:
            mod_mass_delta += MOD_MASSES[mod_name]
        else:
            unknown_modifications.append(mod_name)
            unknown_ptm_counter[mod_name] += 1

    # -------------------------
    # Remove PTM annotations
    # -------------------------
    clean_seq = re.sub(r"\[.*?\]", "", precursor_nocharge)

    clean_seq = re.sub(
        r"[^ACDEFGHIKLMNPQRSTVWY]",
        "",
        clean_seq.upper()
    )

    return {
        "sequence": clean_seq,
        "charge": charge,
        "modifications": modifications,
        "unknown_modifications": unknown_modifications,
        "mod_mass_delta": mod_mass_delta
    }


def compute_properties(seq, mod_mass_delta, charge):

    if len(seq) == 0:
        return None

    analysis = ProteinAnalysis(seq)

    aa_freq = analysis.amino_acids_percent

    canonical_mw = analysis.molecular_weight()

    modified_mw = canonical_mw + mod_mass_delta

    precursor_mz = None

    if charge and charge > 0:
        precursor_mz = (
            modified_mw + charge * PROTON_MASS
        ) / charge

    result = {
        "length": len(seq),
        "mw": canonical_mw,
        "modified_mw": modified_mw,
        "precursor_mz": precursor_mz,
        "gravy": analysis.gravy(),
        "pI": analysis.isoelectric_point(),
    }

    for aa in "ACDEFGHIKLMNPQRSTVWY":
        result[f"freq_{aa}"] = aa_freq.get(aa, 0)

    return result

# ============================================
# LOAD DATA
# ============================================

df = pd.read_csv(INPUT_FILE, sep="\t")

required_columns = [
    "PG.ProteinGroups",
    "EG.PrecursorId"
]

for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

# ============================================
# PROCESS
# ============================================

rows = []

for _, row in tqdm(df.iterrows(), total=len(df)):

    protein_groups = row["PG.ProteinGroups"]
    precursor = row["EG.PrecursorId"]

    parsed = parse_precursor(precursor)

    props = compute_properties(
        parsed["sequence"],
        parsed["mod_mass_delta"],
        parsed["charge"]
    )

    out = {
        "PG.ProteinGroups": protein_groups,
        "EG.PrecursorId": precursor,
        "Sequence": parsed["sequence"],
        "Charge": parsed["charge"],
        "Modifications": "; ".join(parsed["modifications"]),
        "Num_Modifications": len(parsed["modifications"]),
        "Unknown_Modifications": "; ".join(parsed["unknown_modifications"]),
        "Num_Unknown_Modifications": len(parsed["unknown_modifications"]),
        "Modification_Mass_Delta": parsed["mod_mass_delta"],
    }

    if props:
        out.update(props)

    rows.append(out)

# ============================================
# SAVE OUTPUT
# ============================================

output_df = pd.DataFrame(rows)

output_df.to_csv(
    OUTPUT_FILE,
    sep="\t",
    index=False
)

print(f"\nOutput written to: {OUTPUT_FILE}")

# ============================================
# PTM SUMMARY
# ============================================

print("\n" + "=" * 60)
print("PTM SUMMARY")
print("=" * 60)

if len(ptm_counter) == 0:
    print("No PTMs detected.")
else:
    print("\nKnown/Observed PTMs:")
    for ptm, count in ptm_counter.most_common():
        print(f"{ptm}: {count}")

print("\n" + "-" * 60)
print("UNKNOWN PTMs")
print("-" * 60)

if len(unknown_ptm_counter) == 0:
    print("No unknown PTMs found.")
else:
    for ptm, count in unknown_ptm_counter.most_common():
        print(f"{ptm}: {count}")

print("\nTotal unique PTMs found:", len(ptm_counter))
print("Total unique unknown PTMs:", len(unknown_ptm_counter))